# CADI Deep Learning - Semana 13
# # Semana 13: Implementación de un Autoencoder en una Red Denoising usando el dataset MNIST en Google Colab 
# **Estudiante:** Jenifer Cárdenas Aguilera

Para esta actividad armé un autoencoder convolucional que aprende a quitarle ruido a las imágenes de MNIST. La idea es entrenar la red con imágenes "dañadas" como entrada y las imágenes originales como referencia para que aprenda a reconstruirlas.

## 1. Cargar MNIST y agregar ruido

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# normalizar a 0-1
x_train = x_train.astype('float32') / 255.
x_test  = x_test.astype('float32') / 255.

# le agrego el canal porque voy a usar conv2d
x_train = x_train.reshape(-1, 28, 28, 1)
x_test  = x_test.reshape(-1, 28, 28, 1)

print(x_train.shape, x_test.shape)

Para el ruido uso una distribución normal. Probé con factor 0.5 pero quedaba muy difícil de recuperar, así que dejé 0.4.

In [ ]:
factor = 0.4

x_train_ruido = x_train + factor * np.random.normal(0, 1, x_train.shape)
x_test_ruido  = x_test  + factor * np.random.normal(0, 1, x_test.shape)

# recortar al rango valido
x_train_ruido = np.clip(x_train_ruido, 0, 1)
x_test_ruido  = np.clip(x_test_ruido, 0, 1)

In [ ]:
# ver como quedan las imagenes con ruido antes de meterlas al modelo
plt.figure(figsize=(10, 2))
for i in range(6):
    plt.subplot(1, 6, i+1)
    plt.imshow(x_test_ruido[i].reshape(28, 28), cmap='gray')
    plt.axis('off')
plt.show()

## 2. El modelo

Encoder-decoder convolucional. El encoder reduce la imagen a una versión chiquita (cuello de botella) y el decoder la vuelve a estirar al tamaño original.

In [ ]:
def crear_autoencoder_denoising():
    # encoder
    input_img = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(input_img)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(x)
    encoded = layers.MaxPooling2D((2, 2), padding='same')(x)
    
    # decoder
    x = layers.Conv2D(8, (3, 3), activation='relu', padding='same')(encoded)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(16, (3, 3), activation='relu', padding='same')(x)
    x = layers.UpSampling2D((2, 2))(x)
    decoded = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)
    
    return models.Model(input_img, decoded)

autoencoder = crear_autoencoder_denoising()
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
autoencoder.summary()

## 3. Entrenamiento

Entreno con las imagenes ruidosas como X y las originales como Y. Le pongo 10 épocas, con menos no se notaba bien la diferencia.

In [ ]:
historia = autoencoder.fit(
    x_train_ruido, x_train,
    epochs=10,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_ruido, x_test)
)

In [ ]:
# curva de loss
plt.plot(historia.history['loss'], label='train')
plt.plot(historia.history['val_loss'], label='val')
plt.xlabel('epoca')
plt.ylabel('loss')
plt.legend()
plt.show()

## 4. Resultados

Acá comparo las tres versiones lado a lado: la original, la que le metí con ruido y la reconstruida por el modelo.

In [ ]:
recon = autoencoder.predict(x_test_ruido[:10])

n = 5
plt.figure(figsize=(10, 6))
for i in range(n):
    # original
    plt.subplot(3, n, i+1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    if i == 0: plt.title("Original")
    plt.axis('off')
    
    # con ruido
    plt.subplot(3, n, i+1+n)
    plt.imshow(x_test_ruido[i].reshape(28, 28), cmap='gray')
    if i == 0: plt.title("Con ruido")
    plt.axis('off')
    
    # reconstruida
    plt.subplot(3, n, i+1+2*n)
    plt.imshow(recon[i].reshape(28, 28), cmap='gray')
    if i == 0: plt.title("Reconstruida")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# calculo el MSE para tener un numero
from sklearn.metrics import mean_squared_error

recon_full = autoencoder.predict(x_test_ruido, verbose=0)

mse_ruido = mean_squared_error(x_test.reshape(len(x_test), -1),
                               x_test_ruido.reshape(len(x_test), -1))
mse_recon = mean_squared_error(x_test.reshape(len(x_test), -1),
                               recon_full.reshape(len(recon_full), -1))

print("MSE ruidosa vs original:     ", round(mse_ruido, 5))
print("MSE reconstruida vs original:", round(mse_recon, 5))
print("Mejora:", round((1 - mse_recon/mse_ruido)*100, 2), "%")

## 5. Conclusiones

**Sobre el entrenamiento.** La perdida bajó parejo tanto en train como en validación, lo que indica que no hubo sobreajuste fuerte. En la curva se ve que después de la época 6-7 ya casi no mejora, podría haber parado antes.

**Sobre la reconstrucción.** Las imágenes reconstruidas se ven bastante limpias comparado con las versiones con ruido. Pierde un poco de detalle en los trazos finos (los números quedan algo "redondeados") pero sigue siendo facil identificar de que dígito se trata. El MSE confirma que el error se reduce de forma notoria al pasar las imágenes por el modelo.

**Aplicaciones que se me ocurren.** Lo primero que pensé fue restauración de documentos escaneados o fotos viejas. También sirve para imágenes médicas con ruido (radiografías, ecografías). Otra aplicación interesante es la detección de anomalías: si una entrada no se puede reconstruir bien, probablemente sea algo raro que el modelo nunca vio en entrenamiento.

Esta arquitectura es básica pero es la base de cosas más complejas como los VAE o los modelos de difusión, que es a donde apuntan los modelos generativos actuales.